In [1]:
import pandas as pd

In [2]:
file_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Clostridium_difficile.xlsx"
sheet = "test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols, index_col=0
)

In [3]:
df.head()

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,No,Yes,No,Using antibodies (VHHs) AH3 and AA6 are two po...
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,No,Yes,No,"Administration of the PPAR-γ agonist, pioglita..."
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,No,Yes,No,Use Inulin or pectin as a dietary-based therap...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,No,Yes,No,The paper studied a protein named PtsHN10M tha...
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,Yes,No,No,This study focused on analyzing Clostridioides...


In [4]:
df.shape

(50, 6)

In [5]:
from LLM import Clasificador

e:\TuttiQuanti\Trabajo\Anaconda3\envs\env_LLMzCor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
clasificador=Clasificador()

In [11]:
paper=df.loc[34775130, "Abstract"]
clasificador.clasificacion(paper)

"\nAnswer: [2, 'The paper discusses a case series of patients treated with meropenem/vaborbactam (MVB) for serious carbapenem-resistant Enterobacterales (CRE) infections, which falls under the category of new treatments.']"

In [ ]:
#Prueba 

paper=df['Abstract'][7]
clasificador.clasificacion(paper)

def append_answer_to_csv(response: str, csv_path: str):
    """
    Añade directamente la respuesta del clasificador al CSV usando pandas.
    Solo requiere pandas y la función builtin open(), sin re ni os.
    """
    # Crear DataFrame con la respuesta cruda
    df = pd.DataFrame({'Answer': [response]})
 # Abrir el archivo en modo append; si está vacío, escribe la cabecera
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        # f.tell() == 0 indica archivo vacío (sin bytes previos)
        df.to_csv(f, header=(f.tell() == 0), index=False)
# Ejemplo de uso:
if __name__ == '__main__':
    # Supongamos que 'paper' ya está definido y clasificador disponible
    # response es la cadena tal como la devuelve el clasificador
    response = clasificador.clasificacion(paper)

    # Añadir la respuesta al CSV (se creará si no existe)
    append_answer_to_csv(response, 'results.csv')
    print("Respuesta añadida a results.csv")

In [121]:
##prueba agregar PMID

def append_answer_to_csv(pmid: int, response: str, csv_path: str):
    """
    Añade directamente el PMID y la respuesta del clasificador al CSV usando pandas.
    """
    # Crear DataFrame con PMID y respuesta
    df_out = pd.DataFrame({
        'PMID':  [pmid],
        'Answer':[response]
    })
    # Abrir el archivo en modo append; si está vacío, escribe la cabecera
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        df_out.to_csv(f, header=(f.tell() == 0), index=False)

if __name__ == '__main__':
    # 1) Extrae el paper y su PMID
    idx = 7
    pmid  = df['PMID'].iloc[idx]
    paper = df['Abstract'].iloc[idx]
   
    # 2) Obtén la respuesta del clasificador
    response = clasificador.clasificacion(paper)

    # 3) Añade al CSV tanto el PMID como la respuesta
    append_answer_to_csv(pmid, response, 'results.csv')
    print(f"PMID {pmid} y su respuesta añadidos a results.csv")

PMID 35939437 y su respuesta añadidos a results.csv


In [ ]:
# 
# def append_answer_to_csv(response: str, csv_path: str):
#     """
#     Añade directamente la respuesta del clasificador al CSV usando pandas.
#     Solo requiere pandas y la función builtin open(), sin re ni os.
#     """
#     # Crear DataFrame con la respuesta cruda
#     df = pd.DataFrame({'Answer': [response]})
#  # Abrir el archivo en modo append; si está vacío, escribe la cabecera
#     with open(csv_path, 'a', newline='', encoding='utf-8') as f:
#         # f.tell() == 0 indica archivo vacío (sin bytes previos)
#         df.to_csv(f, header=(f.tell() == 0), index=False)
# # Ejemplo de uso:
# if __name__ == '__main__':
#     # Supongamos que 'paper' ya está definido y clasificador disponible
#     # response es la cadena tal como la devuelve el clasificador
#     response = clasificador.clasificacion(paper)

#     # Añadir la respuesta al CSV (se creará si no existe)
#     append_answer_to_csv(response, 'results.csv')
#     print("Respuesta añadida a results.csv")

In [122]:
##Prueba Loop

# --- Bucle en chunks de 10 papers ---
csv_path = 'results.csv'
for start in range(0, len(df), 10):
    batch = df.iloc[start:start+10]
    print(f"Procesando abstracts {start+1} a {start+len(batch)}…")
    for _, row in batch.iterrows():
        pmid     = row['PMID']
        abstract = row['Abstract']
        # Llamas al clasificador para cada abstract
        response = clasificador.clasificacion(abstract)
        # Guardas PMID + respuesta
        append_answer_to_csv(pmid, response, csv_path)

print("¡Terminado! Todas las respuestas están en", csv_path)

Procesando abstracts 1 a 10…
Procesando abstracts 11 a 20…
Procesando abstracts 21 a 30…
Procesando abstracts 31 a 40…
Procesando abstracts 41 a 50…
¡Terminado! Todas las respuestas están en results.csv
